# 🏏 IPL Match Predictor

Machine Learning project to predict the winning probability of an IPL batting team.

In [ ]:
# Step 1: Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries imported successfully!")

In [ ]:
# Step 2: Create a synthetic IPL dataset

np.random.seed(42)

teams = [
    "CSK", "DC", "GT", "KKR", "LSG",
    "MI", "PBKS", "RCB", "RR", "SRH"
]

venues = [
    "Chennai",
    "Delhi",
    "Ahmedabad",
    "Kolkata",
    "Lucknow",
    "Mumbai",
    "Mullanpur",
    "Bengaluru",
    "Jaipur",
    "Hyderabad"
]

home_team = {
    "Chennai": "CSK",
    "Delhi": "DC",
    "Ahmedabad": "GT",
    "Kolkata": "KKR",
    "Lucknow": "LSG",
    "Mumbai": "MI",
    "Mullanpur": "PBKS",
    "Bengaluru": "RCB",
    "Jaipur": "RR",
    "Hyderabad": "SRH"
}

N = 5000

# Generate teams
batting_team = np.random.choice(teams, N)
bowling_team = np.random.choice(teams, N)

# Make sure batting and bowling teams are different
for i in range(N):
    while bowling_team[i] == batting_team[i]:
        bowling_team[i] = np.random.choice(teams)

# Generate match conditions
venue = np.random.choice(venues, N)
target = np.random.randint(130, 221, N)
balls_left = np.random.randint(6, 121, N)
wickets_left = np.random.randint(1, 11, N)

# Generate runs needed
runs_needed = np.array([
    np.random.randint(1, min(target[i], balls_left[i] * 8) + 1)
    for i in range(N)
])

# Team strength values
strength = {
    "CSK": 0.15,
    "DC": 0.02,
    "GT": 0.18,
    "KKR": 0.20,
    "LSG": 0.05,
    "MI": 0.17,
    "PBKS": -0.02,
    "RCB": 0.08,
    "RR": 0.12,
    "SRH": 0.10
}

bat_strength = np.array([
    strength[team] for team in batting_team
])

bowl_strength = np.array([
    strength[team] for team in bowling_team
])

# Home advantage
home_advantage = np.array([
    0.15 if batting_team[i] == home_team[venue[i]] else 0
    for i in range(N)
])

# Required run rate
required_run_rate = runs_needed / balls_left

# Create a realistic probability score
score = (
    0.5
    + bat_strength
    - bowl_strength
    + home_advantage
    + (wickets_left * 0.08)
    + (balls_left * 0.005)
    - (required_run_rate * 0.35)
)

# Convert score to probability using sigmoid
probability = 1 / (1 + np.exp(-score))

# Generate result
# 1 = batting team wins
# 0 = bowling team wins
result = np.random.binomial(1, probability)

# Create DataFrame
df = pd.DataFrame({
    "batting_team": batting_team,
    "bowling_team": bowling_team,
    "venue": venue,
    "runs_needed": runs_needed,
    "wickets_left": wickets_left,
    "balls_left": balls_left,
    "target": target,
    "result": result
})

print("Dataset created successfully!")
print("Dataset shape:", df.shape)
print("\nFirst 10 rows:")
print(df.head(10))

In [ ]:
# Step 3: Check the dataset

print("Missing values:")
print(df.isnull().sum())

print("\nResult distribution:")
print(df["result"].value_counts())

print("\nBatting team win percentage:")
print(f"{df['result'].mean() * 100:.2f}%")

In [ ]:
# Step 4: Select features and target

features = [
    "batting_team",
    "bowling_team",
    "venue",
    "runs_needed",
    "wickets_left",
    "balls_left",
    "target"
]

X = df[features]
y = df["result"]

categorical_features = [
    "batting_team",
    "bowling_team",
    "venue"
]

numeric_features = [
    "runs_needed",
    "wickets_left",
    "balls_left",
    "target"
]

print("Features selected successfully!")
print(features)

In [ ]:
# Step 5: Preprocess categorical data

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

print("Data preprocessing configured successfully!")

In [ ]:
# Step 6: Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# Step 7: Create and train Random Forest model

model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=3,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

model.fit(X_train, y_train)

print("Model trained successfully!")

In [ ]:
# Step 8: Evaluate the model

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("========================================")
print("MODEL EVALUATION")
print("========================================")
print(f"Prediction Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Bowling Team Wins",
            "Batting Team Wins"
        ]
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Step 9: Prediction function

def predict_winner(
    batting_team,
    bowling_team,
    venue,
    runs_needed,
    wickets_left,
    balls_left,
    target
):
    """Predict the winning probability for an IPL match situation."""

    # Validate teams
    if batting_team not in teams:
        raise ValueError(
            f"Invalid batting team: {batting_team}. "
            f"Choose from {teams}"
        )

    if bowling_team not in teams:
        raise ValueError(
            f"Invalid bowling team: {bowling_team}. "
            f"Choose from {teams}"
        )

    if batting_team == bowling_team:
        raise ValueError(
            "Batting team and bowling team cannot be the same."
        )

    # Validate venue
    if venue not in venues:
        raise ValueError(
            f"Invalid venue: {venue}. "
            f"Choose from {venues}"
        )

    # Validate numerical inputs
    if runs_needed < 1:
        raise ValueError("runs_needed must be greater than 0.")

    if wickets_left < 1 or wickets_left > 10:
        raise ValueError("wickets_left must be between 1 and 10.")

    if balls_left < 1 or balls_left > 120:
        raise ValueError("balls_left must be between 1 and 120.")

    if target < 1:
        raise ValueError("target must be greater than 0.")

    if runs_needed > target:
        raise ValueError("runs_needed cannot be greater than target.")

    # Create input DataFrame
    match_data = pd.DataFrame({
        "batting_team": [batting_team],
        "bowling_team": [bowling_team],
        "venue": [venue],
        "runs_needed": [runs_needed],
        "wickets_left": [wickets_left],
        "balls_left": [balls_left],
        "target": [target]
    })

    # Predict probabilities
    probabilities = model.predict_proba(match_data)[0]

    # Classes are [0, 1]
    bowling_win_probability = probabilities[0] * 100
    batting_win_probability = probabilities[1] * 100

    if batting_win_probability >= bowling_win_probability:
        winner = batting_team
    else:
        winner = bowling_team

    print("\n========================================")
    print("       IPL MATCH PREDICTION")
    print("========================================")
    print(f"Batting Team : {batting_team}")
    print(f"Bowling Team : {bowling_team}")
    print(f"Venue        : {venue}")
    print(f"Runs Needed  : {runs_needed}")
    print(f"Wickets Left : {wickets_left}")
    print(f"Balls Left   : {balls_left}")
    print(f"Target       : {target}")
    print("----------------------------------------")
    print(
        f"{batting_team} Winning Probability: "
        f"{batting_win_probability:.2f}%"
    )
    print(
        f"{bowling_team} Winning Probability: "
        f"{bowling_win_probability:.2f}%"
    )
    print("----------------------------------------")
    print(f"Predicted Winner: {winner}")
    print("========================================")

    return winner

In [ ]:
# Step 10: Test prediction

# Example:
# KKR needs 12 runs from 6 balls with 8 wickets remaining
# against CSK at Kolkata.

predict_winner(
    batting_team="KKR",
    bowling_team="CSK",
    venue="Kolkata",
    runs_needed=12,
    wickets_left=8,
    balls_left=6,
    target=145
)

In [ ]:
# Step 11: Another example

predict_winner(
    batting_team="RCB",
    bowling_team="MI",
    venue="Bengaluru",
    runs_needed=35,
    wickets_left=6,
    balls_left=18,
    target=190
)

## Project Summary

The model uses the following match features:

- Batting team
- Bowling team
- Venue
- Runs needed
- Wickets remaining
- Balls remaining
- Target

The Random Forest classifier predicts whether the batting team or bowling team is more likely to win.

**Important:** The current dataset is synthetic and is intended for learning/demo purposes. Real IPL prediction requires historical IPL data.